In [7]:
import pandas as pd

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!pip -q install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 413.9/413.9 kB 33.0 MB/s eta 0:00:00


In [8]:
import optuna
from optuna.pruners import MedianPruner # הוספת הגיזום
from transformers import set_seed
import numpy as np
import itertools
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)
from sklearn.metrics import f1_score, accuracy_score

# -----------------------
# A) Load JSONL
# -----------------------
data_files = {
    "train": "/content/drive/MyDrive/nlp_pro2/data_sets/use_part/train.jsonl",
    "validation": "/content/drive/MyDrive/nlp_pro2/data_sets/use_part/val.jsonl",
    "test": "/content/drive/MyDrive/nlp_pro2/data_sets/use_part/test.jsonl",
}
ds = load_dataset("json", data_files=data_files)

# -----------------------
# B) Label mapping
# -----------------------
label_list = ["entailment", "contradiction", "neutral"]
label2id = {l: i for i, l in enumerate(label_list)}
id2label = {i: l for l, i in label2id.items()}

def add_label_id(example):
    example["label"] = label2id[str(example["label"]).lower()]
    return example

ds = ds.map(add_label_id)

# -----------------------
# C) Tokenizer
# -----------------------
model_name = "onlplab/alephbert-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

if tokenizer.pad_token is None:
    tokenizer.add_special_tokens({"pad_token": "[PAD]"})

def tokenize_pair(example):
    return tokenizer(
        example["translation1"],
        example["translation2"],
        truncation=True,
        max_length=128,
    )

ds_tok = ds.map(tokenize_pair, batched=False)
cols = ["input_ids", "attention_mask", "label"]
ds_tok = ds_tok.remove_columns([c for c in ds_tok["train"].column_names if c not in cols])

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# -----------------------
# D) Metrics
# -----------------------
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro"),
    }

# -----------------------
# E) Model init (fresh model per run)
# -----------------------
def model_init():
    m = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=3,
        id2label=id2label,
        label2id=label2id,
    )
    # if we added pad token, keep embeddings consistent
    if len(tokenizer) != m.config.vocab_size:
        m.resize_token_embeddings(len(tokenizer))
    return m


set_seed(42)

def hp_space(trial: optuna.Trial):
    return {
        "learning_rate": trial.suggest_float("learning_rate", 2e-5, 5e-5, log=True),
        "weight_decay": trial.suggest_float("weight_decay", 0.01, 0.1),
        "warmup_ratio": trial.suggest_float("warmup_ratio", 0.0, 0.1),
        "num_train_epochs": trial.suggest_int("num_train_epochs", 2, 3),
        "per_device_train_batch_size": 32, # קיבוע ל-32 לחיסכון בזמן (A100 מתמודד עם זה בקלות)
    }

def compute_objective(metrics):
    # HF will pass eval metrics dict here
    return metrics["eval_macro_f1"]

# --- עדכון ה-TrainingArguments הבסיסיים ---
base_args = TrainingArguments(
    output_dir="hebert_optuna",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    report_to="none",
    seed=42,
    fp16=True, # האצה משמעותית ב-A100
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    per_device_eval_batch_size=32, # Batch קבוע להערכה מהירה
)

trainer = Trainer(
    args=base_args,
    model_init=model_init,
    train_dataset=ds_tok["train"],
    eval_dataset=ds_tok["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=1)],
)


# הפעלת החיפוש עם Pruner
best_run = trainer.hyperparameter_search(
    backend="optuna",
    direction="maximize",
    hp_space=hp_space,
    compute_objective=compute_objective,
    n_trials=10, # 10 ניסויים מספיקים לרוב עם Bayesian Optimization
    pruner=MedianPruner(), # עוצר ניסויים חלשים באמצע
)

print("Best trial:", best_run)
best_cfg = best_run.hyperparameters
print("BEST CFG:", best_cfg)


# -----------------------
# G) FINAL TRAIN with best hyperparameters (then test ONCE)
# -----------------------
final_args = TrainingArguments(
    output_dir="hebert_final",
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    learning_rate=best_cfg["learning_rate"],
    weight_decay=best_cfg["weight_decay"],
    warmup_ratio=best_cfg["warmup_ratio"],
    per_device_train_batch_size=best_cfg["per_device_train_batch_size"],
    per_device_eval_batch_size=32,
    num_train_epochs=best_cfg["num_train_epochs"],
    report_to="none",
    seed=42,
)

final_trainer = Trainer(
    args=final_args,
    model_init=model_init,
    train_dataset=ds_tok["train"],
    eval_dataset=ds_tok["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

final_trainer.train()

print("\nTEST:")
test_metrics = final_trainer.evaluate(ds_tok["test"])
print(test_metrics)


Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/99999 [00:00<?, ? examples/s]

Map:   0%|          | 0/21429 [00:00<?, ? examples/s]

...

[I 2026-02-01 00:11:43,100] Trial 8 pruned. 
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at onlplab/alephbert-base and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,0.811700,0.690329,0.712492,0.710839
2,0.527600,0.689087,0.733912,0.733691


[I 2026-02-01 00:17:24,102] Trial 9 finished with value: 0.7336907672546618 and parameters: {'learning_rate': 3.76690649380035e-05, 'weight_decay': 0.039841378722297226, 'warmup_ratio': 0.09881535889557601, 'num_train_epochs': 2}. Best is trial 9 with value: 0.7336907672546618.


Best trial: BestRun(run_id='9', objective=0.7336907672546618, hyperparameters={'learning_rate': 3.76690649380035e-05, 'weight_decay': 0.039841378722297226, 'warmup_ratio': 0.09881535889557601, 'num_train_epochs': 2}, run_summary=None)
BEST CFG: {'learning_rate': 3.76690649380035e-05, 'weight_decay': 0.039841378722297226, 'warmup_ratio': 0.09881535889557601, 'num_train_epochs': 2}


KeyError: 'per_device_train_batch_size'

BEST CFG:
*  'learning_rate': 3.76690649380035e-05,
* 'weight_decay': 0.039841378722297226,
* 'warmup_ratio': 0.09881535889557601,